# GOBR — execução no Google Colab (GPU)

Este notebook **não substitui o projeto no Windows**. Ele baixa a mesma codebase do GitHub, instala as dependências, usa o Ollama **dentro da máquina do Colab** e executa a primeira entrega e a geração incremental de explicações (parser → interpretador → explicador).

**O que fica onde**
- **Windows:** edição da codebase e commits.
- **GitHub:** versão oficial do código para a disciplina.
- **Colab:** execução temporária com GPU (quando disponível).
- **Drive:** resultados das explicações.

O Colab gratuito tem limites variáveis de GPU/tempo e pode desconectar. **Não é um servidor permanente**. O Ollama não será exposto à internet.

**Antes de executar:** envie a pasta do projeto para um repositório no GitHub. Use um repositório público para o comando de clone mostrado aqui. Para repositório privado, não coloque tokens no código do notebook; use a alternativa de upload de ZIP ou configure autenticação de maneira segura.

## 1. Ative a GPU
No menu do Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: T4 GPU** (ou outra GPU disponível). Depois clique no botão ▶ desta célula.

A GPU deve aparecer em `nvidia-smi`. Se não aparecer, volte às configurações do ambiente e escolha uma GPU.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

## 2. Monte o Google Drive
O botão ▶ abrirá uma janela pedindo permissão para o Colab acessar seu Drive. As saídas serão salvas em `Meu Drive/gobr-agent/outputs/`. O código fica no GitHub, não no Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Baixe a mesma codebase do GitHub
Substitua a URL abaixo pela URL **real** do seu repositório público no GitHub. Exemplo: `https://github.com/usuario/sistemas-multiagentes.git`.

Sempre que você alterar o projeto no Windows e executar `git push`, execute novamente esta célula no Colab para baixar a nova versão. **Não edite agentes no Colab**, pois as alterações são temporárias e não voltam automaticamente ao GitHub.

**Alternativa:** se ainda não publicou seu repositório, altere `ORIGEM` para `"zip"`; o Colab solicitará o upload de um ZIP do projeto, sem enviar tokens de acesso.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

ORIGEM = "github"  # "github" ou "zip"
REPO_URL = "https://github.com/gsilverio/sistemas-multiagentes.git"
BRANCH = "main"  # branch com o codigo a executar
PROJETO = Path('/content/gobr-project')

if any(nome == 'src' or nome.startswith('src.') for nome in sys.modules):
    raise RuntimeError(
        'O código antigo ainda está carregado na memória. '
        'Reinicie a sessão do Colab antes de clonar a nova revisão.'
    )

if PROJETO.exists():
    shutil.rmtree(PROJETO)  # Apenas a copia temporaria do Colab

if ORIGEM == "github":
    if "SEU_USUARIO" in REPO_URL or "SEU_REPOSITORIO" in REPO_URL:
        raise ValueError("Troque REPO_URL pela URL real do seu repositorio GitHub.")
    if not REPO_URL.startswith("https://github.com/"):
        raise ValueError("Use uma URL HTTPS de repositorio github.com.")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJETO)], check=True)
elif ORIGEM == "zip":
    from google.colab import files
    enviados = files.upload()
    if len(enviados) != 1:
        raise ValueError("Envie somente um arquivo ZIP da codebase.")
    nome_zip = next(iter(enviados))
    PROJETO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(nome_zip) as pacote:
        for member in pacote.infolist():
            destino = (PROJETO / member.filename).resolve()
            if not destino.is_relative_to(PROJETO.resolve()):
                raise ValueError(f"Caminho inseguro no ZIP: {member.filename}")
        pacote.extractall(PROJETO)
else:
    raise ValueError("ORIGEM deve ser 'github' ou 'zip'.")

if not (PROJETO / 'src' / 'gobr' / 'parser.py').is_file():
    raise FileNotFoundError("O projeto deve conter src/gobr/parser.py.")

os.chdir(PROJETO)
sys.path.insert(0, str(PROJETO))
print('Projeto carregado em:', PROJETO)
print('Arquivos de regras:', len(list((PROJETO / 'data' / 'referencias').glob('*.gobr'))))

## 4. Instale as dependências Python
Lê o mesmo `requirements.txt` versionado no GitHub. Esta instalação precisa ser refeita quando o Colab criar uma nova máquina virtual.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJETO / 'requirements.txt')],
    check=True
)
print('Dependencias instaladas.')

## 5. Instale e inicie o Ollama **somente no Colab**
O Ollama executa o modelo dentro da máquina virtual do Colab. O servidor estará acessível **apenas pelo próprio notebook** em `127.0.0.1:11434`, não pelo Windows.

O script de instalação abaixo é o procedimento fornecido no site oficial do Ollama.

In [ ]:
import shutil

if shutil.which('ollama'):
    print('Ollama já está instalado:', shutil.which('ollama'))
else:
    subprocess.run([
        'bash', '-lc',
        'set -euo pipefail; ' 
        'if ! command -v zstd >/dev/null 2>&1; then apt-get update -qq; apt-get install -y -qq zstd; fi; ' 
        'curl -fsSL https://ollama.com/install.sh -o /tmp/ollama-install.sh; ' 
        'bash /tmp/ollama-install.sh',
    ], check=True)
    print('Ollama instalado.')

In [ ]:
import time
import urllib.request

os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'  # mantém o modelo carregado entre as chamadas
LOG_OLLAMA = '/tmp/ollama-gobr.log'

def ollama_pronto():
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

def iniciar_ollama():
    """Inicia `ollama serve` se ele não estiver respondendo e espera ficar pronto."""
    if ollama_pronto():
        return
    log = open(LOG_OLLAMA, 'a')
    subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(60):
        if ollama_pronto():
            return
        time.sleep(1)
    raise RuntimeError(f'Ollama nao iniciou. Consulte {LOG_OLLAMA}')

def reiniciar_ollama():
    subprocess.run(['pkill', '-f', 'ollama serve'])
    time.sleep(3)
    iniciar_ollama()

def log_ollama(linhas=25):
    if os.path.exists(LOG_OLLAMA):
        print(''.join(open(LOG_OLLAMA, encoding='utf-8', errors='replace').readlines()[-linhas:]))

iniciar_ollama()
print('Ollama pronto em http://127.0.0.1:11434')

## 6. Baixe o modelo maior
Começaremos com `gemma3:12b` (~8,2 GB de download de modelo). Ele deverá caber em muitas GPUs T4 de 16 GB, mas o uso real depende do contexto e da memória disponível. **Baixar o modelo não garante que a inferência use inteiramente a GPU.**

Se houver erro de memória, podemos testar um modelo menor. O modelo pode ser alterado aqui sem modificar os agentes.

In [ ]:
MODELO = 'gemma3:12b'
CONTEXTO = 4096

processo_modelo = subprocess.Popen(
    ['ollama', 'pull', MODELO],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for linha in processo_modelo.stdout:
    print(linha, end='', flush=True)
codigo = processo_modelo.wait()
if codigo != 0:
    raise subprocess.CalledProcessError(codigo, ['ollama', 'pull', MODELO])
os.environ['GOBR_LLM_MODEL'] = MODELO
os.environ['GOBR_LLM_NUM_CTX'] = str(CONTEXTO)
os.environ['OLLAMA_BASE_URL'] = 'http://127.0.0.1:11434'
print('Modelo preparado:', MODELO)

## 7. Confira se o Ollama realmente usou a GPU
O comando abaixo mostra os modelos carregados e o dispositivo usado. Se aparecer processamento misto GPU/CPU, pode significar memória de vídeo insuficiente para manter todo o modelo na GPU.

In [ ]:
# PROCESSOR deve mostrar "100% GPU". Se aparecer "CPU/GPU", parte do modelo está na CPU:
# fica bem mais lento e mais sujeito a queda por falta de memória.
print(subprocess.run(['ollama', 'ps'], capture_output=True, text=True).stdout)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

## Quando você voltar amanhã
1. Se o código no GitHub mudou, reinicie a sessão do Colab (menu **Ambiente de execução → Reiniciar sessão**) antes de clonar, para evitar módulos Python antigos em memória.
2. Abra a versão atualizada deste notebook no GitHub.
3. Se necessário, ative a GPU e reconecte o Drive.
4. Execute as células de cima para baixo: a máquina virtual anterior pode ter sido apagada.
5. Os arquivos salvos no Drive continuam lá. O código original continua no GitHub.

Referências oficiais: [Google Colab — FAQ](https://research.google.com/colaboratory/intl/pt-BR/faq.html) e [Ollama — instalação Linux](https://ollama.com/download/linux).

## 8. Preparar as 1.700 regras extraídas

Se você acabou de abrir o notebook nesta sessão, execute antes as **etapas 1 a 6** (GPU, Drive, GitHub, dependências, Ollama e modelo).

A pasta de saída no Drive leva o nome do modelo e da **versão do explicador** (um hash dos arquivos que mudam o texto: glossário, rascunho, guard, prompt e agente). Commits que não mexem no explicador continuam na mesma pasta, e a geração retoma de onde parou.

In [ ]:
import sys
import subprocess
from pathlib import Path

if not (PROJETO / 'src' / 'agents' / 'explicador.py').is_file():
    raise FileNotFoundError('Atualize o repositório no GitHub, reinicie a sessão do Colab e clone novamente.')

subprocess.run([sys.executable, '-m', 'src.pipelines.prepare_references'], cwd=str(PROJETO), check=True)
ENTRADA_REGRAS = PROJETO / 'outputs' / 'generated' / 'regras_extraidas.jsonl'
assert ENTRADA_REGRAS.is_file()

VERSAO_EXPLICADOR = subprocess.run(
    [sys.executable, '-c', 'from src.pipelines.generate_explanations import versao_explicador; print(versao_explicador())'],
    cwd=str(PROJETO), capture_output=True, text=True, check=True,
).stdout.strip()
NOME_MODELO = MODELO.replace(':', '_').replace('/', '_')
DESTINO_LOTE = Path('/content/drive/MyDrive/gobr-agent/outputs/explicacoes') / f'{NOME_MODELO}_{VERSAO_EXPLICADOR}'
DESTINO_LOTE.mkdir(parents=True, exist_ok=True)
print('Entrada:', ENTRADA_REGRAS)
print('Saída permanente no Drive:', DESTINO_LOTE)

## 9. Prévia do lote (sem executar o modelo)

Mostra os IDs e nomes das próximas regras. Não consome GPU.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.pipelines.generate_explanations',
    '--input', str(ENTRADA_REGRAS),
    '--output-dir', str(DESTINO_LOTE),
    '--limit', '3', '--dry-run',
], cwd=str(PROJETO), check=True)

## 10. Primeiro lote: cinco explicações

Para cada regra o explicador monta um **rascunho determinístico** (sem LLM), faz **uma chamada** ao modelo para reescrever o texto e confere o resultado com o **guard**. Se o guard reprovar, faz uma chamada de correção com os problemas apontados; se reprovar de novo, usa o rascunho. Cada regra vira `items/<id>.json` no Drive assim que termina.

O método de cada item: `llm` (aprovado de primeira), `llm_corrigido` (aprovado na correção) ou `rascunho` (o modelo errou duas vezes). Blocos iguais a um já explicado aparecem como `reaproveitada de ...`.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'src.pipelines.generate_explanations',
    '--input', str(ENTRADA_REGRAS),
    '--output-dir', str(DESTINO_LOTE),
    '--limit', '5',
], cwd=str(PROJETO), check=True)

## 11. Examinar os resultados

Mostra o GOBR, o texto final, o método e, quando o modelo foi reprovado, o que o guard apontou.

In [ ]:
import json

for arquivo in sorted((DESTINO_LOTE / 'items').glob('*.json'))[:8]:
    item = json.loads(arquivo.read_text(encoding='utf-8'))
    explicacao = item.get('explicacao') or {}
    print('\n---', item['id'], '|', item['status'], '|', explicacao.get('metodo', '-'),
          '| aprovada' if explicacao.get('aprovada') else '')
    print(item['gobr'])
    print('>>', explicacao.get('texto') or item.get('reason') or str(item.get('error', ''))[:400])
    if explicacao.get('metodo') != 'llm' and explicacao.get('tentativas_llm'):
        print('   última tentativa do modelo:', explicacao['tentativas_llm'][-1][:300])
        print('   problemas:', explicacao.get('problemas'))

## 12. Gerar o restante (retomada automática)

Ative `EXECUTAR_MAIS = True` depois de conferir o primeiro lote. A célula processa `LOTE` regras novas e **pula as já concluídas**; execute de novo para o próximo lote. Com `gemma3:12b` numa T4, conte com 5 a 10 s por regra. Blocos repetidos (só o nome muda) reaproveitam a explicação sem chamar o modelo: são cerca de 1.200 chamadas, ≈ 2 a 3 horas no total. Se o Colab desconectar, nada se perde: refaça as etapas 1 a 8 e rode esta célula de novo.

`LOTE = 0` processa tudo o que falta numa só execução. Se o Ollama parar de responder, o lote para sem marcar as regras como falhas, a célula mostra o log, reinicia o Ollama e continua (até `MAX_REINICIOS` vezes).

In [ ]:
EXECUTAR_MAIS = False
LOTE = 0            # 0 = tudo o que falta
MAX_REINICIOS = 5   # quantas vezes reiniciar o Ollama se ele cair no meio

COMANDO = [sys.executable, '-m', 'src.pipelines.generate_explanations',
           '--input', str(ENTRADA_REGRAS), '--output-dir', str(DESTINO_LOTE), '--limit', str(LOTE)]

if EXECUTAR_MAIS:
    for reinicio in range(MAX_REINICIOS + 1):
        iniciar_ollama()
        resultado = subprocess.run(COMANDO, cwd=str(PROJETO))
        if resultado.returncode == 0:
            print('Lote concluído.')
            break
        if resultado.returncode != 3:  # 3 = Ollama parou de responder
            raise subprocess.CalledProcessError(resultado.returncode, COMANDO)
        print(f'\nOllama caiu (reinício {reinicio + 1} de {MAX_REINICIOS}). Últimas linhas do log:')
        log_ollama()
        reiniciar_ollama()
    else:
        raise RuntimeError('O Ollama caiu várias vezes. Veja o log acima e a etapa 7 (memória da GPU).')
else:
    print('Desativado. Altere EXECUTAR_MAIS para True depois de revisar o primeiro lote.')

## 12b. Avaliar as explicações e comparar com a linha de base

`explicacoes.jsonl` (no Drive, dentro de `DESTINO_LOTE`) é atualizado ao fim de cada lote. Esta célula confere todas as explicações com o guard e compara com as 59 explicações do pipeline antigo.

**Quando terminar as 1.700**, baixe `explicacoes.jsonl` do Drive e coloque no projeto em `data/explicacoes/explicacoes_v2.jsonl`: ele alimenta o RAG (etapa 3) e o fine-tuning (etapa 6).

In [ ]:
for arquivo in [DESTINO_LOTE / 'explicacoes.jsonl', PROJETO / 'data' / 'avaliacao' / 'explicacoes_baseline_v1.jsonl']:
    print('=' * 20, arquivo.name)
    subprocess.run([sys.executable, '-m', 'src.pipelines.avaliar_explicacoes', str(arquivo), '--exemplos', '3'],
                   cwd=str(PROJETO), check=True)
resumo = json.loads((DESTINO_LOTE / 'batch_summary.json').read_text(encoding='utf-8'))
print('Último lote:', resumo['metodos_nesta_execucao'], '| falhas:', resumo['falhas_nesta_execucao'])

## 12c. Diagnóstico do lote

Contagem por status e método, tempo médio por chamada ao modelo e as primeiras falhas.

In [ ]:
import collections

itens = [json.loads(a.read_text(encoding='utf-8')) for a in (DESTINO_LOTE / 'items').glob('*.json')]
status = collections.Counter(i['status'] for i in itens)
metodos = collections.Counter((i.get('explicacao') or {}).get('metodo') for i in itens if i['status'] == 'completed')
reaproveitadas = sum(1 for i in itens if i.get('reaproveitada_de'))
tempos = [i['elapsed_seconds'] for i in itens if i['status'] == 'completed' and not i.get('reaproveitada_de')]
print('Itens:', len(itens), '| status:', dict(status))
print('Métodos:', dict(metodos), '| reaproveitadas:', reaproveitadas)
if tempos:
    print(f'Tempo médio por chamada: {sum(tempos)/len(tempos):.1f}s em {len(tempos)} regras')
for i in [i for i in itens if i['status'] == 'failed'][:3]:
    print('FALHA', i['id'], '->', i['error']['type'], i['error']['message'][:200])

## 12d. Descrições no estilo das tabelas dos clientes (dica do enunciado)

O enunciado sugere traduzir as regras `.gobr` em descrições **parecidas com as das imagens** (linhas curtas como "Clientes com compra entre 0 e 60 dias | 14 dias"). As explicações da etapa 12 são completas, mas formais. Este lote gera, para cada corpo distinto de regra (1.202), uma descrição curta nesse estilo, usando como modelo as próprias tabelas de `data/exemplos`. Cada descrição é conferida: números, valores entre aspas, "não agendar" e frequência precisam bater com a regra; se não baterem, o modelo corrige uma vez e, se ainda falhar, o bloco fica sem descrição.

Elas entram no **índice de exemplos (RAG)** e no **dataset do fine-tuning** (etapa 17 usa direto do Drive). Requer o Ollama (etapas 1 a 7, com a etapa 8 para `NOME_MODELO`). Conte cerca de 1 a 1,5 hora numa T4; se a sessão cair, rode de novo e o lote continua de onde parou.

Ao terminar, baixe `descricoes_cliente.jsonl` (caminho impresso abaixo) para `data/explicacoes/descricoes_cliente.jsonl` no repositório local.

In [ ]:
DESTINO_DESCRICOES = Path('/content/drive/MyDrive/gobr-agent/outputs/descricoes_cliente') / NOME_MODELO.replace(':', '_')
COMANDO_DESC = [sys.executable, '-m', 'src.pipelines.gerar_descricoes_cliente', '--output-dir', str(DESTINO_DESCRICOES)]

for reinicio in range(MAX_REINICIOS + 1 if 'MAX_REINICIOS' in globals() else 6):
    iniciar_ollama()
    resultado = subprocess.run(COMANDO_DESC, cwd=str(PROJETO))
    if resultado.returncode == 0:
        print('Lote de descrições concluído:', DESTINO_DESCRICOES / 'descricoes_cliente.jsonl')
        break
    if resultado.returncode != 3:
        raise subprocess.CalledProcessError(resultado.returncode, COMANDO_DESC)
    print('Ollama caiu; reiniciando...')
    log_ollama()
    reiniciar_ollama()

import collections
import json
itens = [json.loads(a.read_text(encoding='utf-8')) for a in (DESTINO_DESCRICOES / 'items').glob('*.json')]
print('Itens:', len(itens), dict(collections.Counter(i.get('metodo') or i['status'] for i in itens)))
for i in itens[:8]:
    print('-', i['id'], '|', i.get('descricao') or i.get('problemas'))

## 13. Recuperação (etapa 3): manual e exemplos

Os índices já vêm no repositório (`outputs/rag/manual` e `outputs/rag/exemplos`): busca **híbrida**, BM25 + embeddings `multilingual-e5-small`, fundidos por RRF. O manual é fatiado **por seção** (ex.: `6.11.1 Opções para tempo`) e os exemplos são as regras do acervo com a explicação. Esta célula não usa o Ollama e roda em CPU; serve para conferir a recuperação.

Não é preciso reconstruir os índices aqui. Quando o lote da etapa 12 terminar, baixe `explicacoes.jsonl` (passo 12b); os índices de exemplos são refeitos no computador local com as explicações do LLM.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers>=3,<6'], check=True)
subprocess.run([sys.executable, '-m', 'src.pipelines.avaliar_recuperacao'], cwd=str(PROJETO), check=True)

from src.rag.recuperador import buscar_exemplos, buscar_manual

consulta = 'clientes que compraram nos últimos 90 dias mas estão há 60 sem comprar, a cada 7 dias'
for trecho in buscar_manual(consulta, k=3):
    print('MANUAL', trecho.secao, trecho.titulo, '|', trecho.citacao())
for exemplo in buscar_exemplos(consulta, k=3):
    print('\nEXEMPLO', exemplo.id, '|', exemplo.explicacao)
    print(exemplo.gobr)

## 14. Grafo multiagente (etapa 4): os 3 exemplos reais

Executa o fluxo completo (Percepção → Analista → Recuperador → Gerador ⇄ Validador → Explicador) sobre uma regra comercial de `data/exemplos`, **sem pausar para perguntas** (`interativo=False`): perguntas não bloqueantes viram suposições registradas na resposta.

**Requer** o Ollama rodando com o modelo (etapas 1 a 7). **Não rode enquanto a etapa 12 estiver gerando explicações**: os dois disputam a mesma GPU e o lote fica mais lento. Rode depois que o lote terminar, ou numa sessão separada.

- `ORQUESTRADOR_COM_LLM = True`: o LLM escolhe o próximo agente quando há mais de uma opção válida (os guardrails corrigem escolhas inválidas). `False` usa só as regras.
- `EXPLICAR_COM_LLM = False`: as explicações usam o rascunho determinístico (rápido). `True` chama o LLM por bloco (mais natural, alguns minutos a mais).

A saída mostra cada decisão do orquestrador, a resposta final e uma comparação com o GOBR escrito à mão para o mesmo cliente.

In [ ]:
import uuid
from IPython.display import Markdown, display
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from src.agents.percepcao import ler_anexo
from src.grafo.estado import Agentes
from src.grafo.grafo import config_thread, construir_grafo, entrada
from src.validation.gobr_validator import GobrValidator

ORQUESTRADOR_COM_LLM = True
EXPLICAR_COM_LLM = False
EXEMPLO = 'RegraMaxlub'   # ou 'RegraInfinittoDesign', 'V1Vidros-RegraComercial'

agentes = Agentes.padrao(orquestrador_com_llm=ORQUESTRADOR_COM_LLM, explicar_com_llm=EXPLICAR_COM_LLM,
                         interativo=False)
grafo = construir_grafo(agentes, checkpointer=InMemorySaver())

texto, imagens = ler_anexo(PROJETO / 'data' / 'exemplos' / f'{EXEMPLO}.pdf')
inicio = time.time()
saida = grafo.invoke(entrada(texto, imagens), config_thread(uuid.uuid4().hex))
print(f'Tempo total: {time.time() - inicio:.0f}s\n')
for passo in saida['trace']:
    if passo['no'] == 'orquestrador':
        print(f"→ {passo['proximo']:18s} [{passo['origem']}] {passo['motivo']}")
display(Markdown(saida['resposta']))

referencia = (PROJETO / 'data' / 'exemplos' / f'{EXEMPLO}.gobr').read_text(encoding='utf-8')
print('\n===== GOBR escrito à mão para este cliente (referência) =====\n')
print(referencia)
print(GobrValidator().validar(saida.get('gobr') or '').como_texto())

## 15. Conversa interativa

Mesmo grafo, mas **pausando para perguntar** quando falta informação (`interrupt()` do LangGraph). Digite a descrição das regras; responda às perguntas ou aperte Enter para seguir com as suposições. Depois da resposta, rode a última parte da célula com um pedido de ajuste (ex.: "mude a frequência dos ativos para 7 dias"): a conversa continua no mesmo `thread_id`.

In [ ]:
agentes_conversa = Agentes.padrao(orquestrador_com_llm=ORQUESTRADOR_COM_LLM, explicar_com_llm=EXPLICAR_COM_LLM,
                                 interativo=True)
grafo_conversa = construir_grafo(agentes_conversa, checkpointer=InMemorySaver())
config = config_thread('conversa-1')

def conversar(mensagem):
    saida = grafo_conversa.invoke(entrada(mensagem), config)
    while '__interrupt__' in saida:
        print(saida['__interrupt__'][0].value['mensagem'])
        saida = grafo_conversa.invoke(Command(resume=input('\nSua resposta (Enter = seguir com as suposições): ')), config)
    display(Markdown(saida['resposta']))
    return saida

saida = conversar(input('Descreva as regras comerciais: '))

In [ ]:
# Ajuste na mesma conversa (o grafo refaz o ciclo com o histórico):
saida = conversar(input('Pedido de ajuste: '))

## 16. Interface Gradio (etapa 5, entrega final)

Sobe o chat do agente com um **link público** (`share=True`, válido enquanto esta sessão do Colab estiver ativa). Requer o Ollama com o modelo (etapas 1 a 7) e, como a etapa 14, não deve rodar junto com o lote da etapa 12.

- Anexe o PDF ou a imagem da tabela, ou descreva as regras. Os 3 exemplos reais aparecem como atalhos.
- Quando o agente perguntar algo, responda no chat ou clique em **Seguir com as suposições**.
- Depois da resposta, peça ajustes na mesma conversa.
- **Memória**: as conversas ficam em `conversas.sqlite` no Drive. Em outra sessão, abra *Conversas salvas → Retomar* e continue de onde parou, inclusive se ela estava pausada numa pergunta.

Se o SQLite no Drive der erro de bloqueio (`database is locked`), troque `BANCO` por `Path('/content/conversas.sqlite')`: funciona igual, mas some quando a sessão acaba.

In [ ]:
from src.grafo.estado import Agentes
from src.interface.app import criar_app
from src.interface.servico import ServicoConversa

ORQUESTRADOR_COM_LLM = True
EXPLICAR_COM_LLM = False
BANCO = Path('/content/drive/MyDrive/gobr-agent/conversas.sqlite')

servico = ServicoConversa(Agentes.padrao(orquestrador_com_llm=ORQUESTRADOR_COM_LLM,
                                         explicar_com_llm=EXPLICAR_COM_LLM), banco=BANCO)
app = criar_app(servico, exemplos=sorted((PROJETO / 'data' / 'exemplos').glob('*.pdf')))
app.queue().launch(share=True)

## 16b. LangGraph Studio: ver e executar o grafo nó a nó

O **LangGraph Studio** é uma página web que mostra o desenho do grafo (os nós e as ligações) e deixa você executar o agente acompanhando cada nó: qual agente rodou, o que ele devolveu e como ficou o estado. Quando o agente faz uma pergunta (`interrupt`), o Studio mostra a pergunta e uma caixa para responder.

Como funciona: esta célula sobe um servidor do LangGraph (`langgraph dev`) dentro do Colab e cria um **túnel** público temporário até ele. Ela imprime um link `https://smith.langchain.com/studio/?baseUrl=...`; abra no seu navegador (Chrome, de preferência). Se o site pedir, entre com uma conta gratuita do LangSmith.

No Studio:
1. Escolha o grafo **agente_gobr** (canto superior esquerdo).
2. Em **Input**, preencha `pedido` com a descrição das regras e, se quiser, `anexo` com `data/exemplos/RegraMaxlub.pdf` (ou outro PDF/imagem do projeto). Clique em **Submit**.
3. Os nós acendem à medida que executam; clique num nó ou passo para ver o estado.
4. Se aparecer uma interrupção, escreva a resposta e retome. Para pedir um ajuste, envie outro `pedido` na mesma thread.

O servidor roda num ambiente Python separado (`/content/studio-venv`) porque o `langgraph-api` exige versões de bibliotecas que conflitam com o Gradio. Requer o Ollama (etapas 1 a 7). Não rode junto com o lote da etapa 12. Para parar: `studio.terminate()`.

In [ ]:
import re

STUDIO_VENV = Path('/content/studio-venv')
LOG_STUDIO = '/tmp/langgraph-studio.log'
if not (STUDIO_VENV / 'bin' / 'langgraph').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', '-q', '--system-site-packages', str(STUDIO_VENV)], check=True)
    subprocess.run([str(STUDIO_VENV / 'bin' / 'pip'), 'install', '-q', 'langgraph-cli[inmem]'], check=True)

os.environ['GOBR_ORQUESTRADOR_LLM'] = '1'   # '0' = orquestrador só com as regras
os.environ['GOBR_EXPLICAR_LLM'] = '0'       # '1' = explicações com o LLM
iniciar_ollama()

if 'studio' in globals() and studio.poll() is None:
    studio.terminate()
studio = subprocess.Popen(
    [str(STUDIO_VENV / 'bin' / 'langgraph'), 'dev', '--tunnel', '--no-browser', '--allow-blocking', '--port', '2024'],
    cwd=str(PROJETO), stdout=open(LOG_STUDIO, 'w'), stderr=subprocess.STDOUT, env=os.environ.copy())

for _ in range(90):
    texto = open(LOG_STUDIO, encoding='utf-8', errors='replace').read()
    tunel = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', texto)
    if tunel and 'Application started' in texto:
        print('Abra no navegador:\n')
        print(f'https://smith.langchain.com/studio/?baseUrl={tunel.group(0)}')
        break
    if studio.poll() is not None:
        print(texto[-3000:])
        raise RuntimeError('O servidor do Studio parou. Veja o log acima.')
    time.sleep(2)
else:
    print(texto[-3000:])
    print('Ainda sem o link do túnel; rode esta célula de novo em alguns segundos.')

## 17. Fine-tuning do Gerador (etapa 6): dataset

Pares **descrição → bloco GOBR** tirados do acervo, no mesmo formato de prompt que o Gerador usa no grafo. Treino e teste são separados por *forma de regra* (nenhuma forma do teste aparece no treino), e os 3 exemplos reais de `data/exemplos` formam um conjunto à parte. Se o lote da etapa 12 já terminou, as explicações do LLM entram como descrições extras no treino.

Ordem das etapas 17 a 21: dataset → avaliar o modelo base → treinar → avaliar o treinado → comparar. Tudo fica no Drive em `FT_DIR`; se a sessão cair, rode a 17 de novo e continue da etapa em que parou (avaliação e treino retomam sozinhos).

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft>=0.13', 'bitsandbytes>=0.45',
                'accelerate>=1.0', 'datasets>=3.0'], check=True)

FT_DIR = Path('/content/drive/MyDrive/gobr-agent/finetuning')
DATASET_FT = FT_DIR / 'dataset'
AVALIACAO_FT = FT_DIR / 'avaliacao'
ADAPTADOR_DIR = FT_DIR / 'qwen3-4b-gobr'
MODELO_BASE_FT = 'Qwen/Qwen3-4B'

# As explicações da etapa 12 já estão no repositório (data/explicacoes/explicacoes_v2.jsonl).
explicacoes_llm = globals().get('DESTINO_LOTE')
explicacoes_llm = explicacoes_llm / 'explicacoes.jsonl' if explicacoes_llm else None
comando = [sys.executable, '-m', 'src.finetuning.dataset', '--saida', str(DATASET_FT)]
if explicacoes_llm and explicacoes_llm.exists():
    comando += ['--explicacoes', str(explicacoes_llm)]
    print('Usando as explicações do LLM:', explicacoes_llm)
else:
    print('Usando as explicações do repositório (data/explicacoes/explicacoes_v2.jsonl).')
descricoes_drive = globals().get('DESTINO_DESCRICOES')
if descricoes_drive and (descricoes_drive / 'descricoes_cliente.jsonl').exists():
    comando += ['--descricoes', str(descricoes_drive / 'descricoes_cliente.jsonl')]
    print('Usando as descrições no estilo das tabelas:', descricoes_drive)
subprocess.run(comando, cwd=str(PROJETO), check=True)

## 18. Avaliar o modelo base (antes do treino)

**Libera a GPU**: o Ollama é parado (as etapas 14 a 16 precisam dele de novo depois; basta rodar a etapa 5). Avalia o `Qwen3-4B` sem treino nos conjuntos de teste e real, com e sem exemplos no prompt (`--com-exemplos` imita o Recuperador). Conte uns 10 a 20 minutos.

In [ ]:
subprocess.run(['pkill', '-f', 'ollama serve'])

AVALIAR_SEM_EXEMPLOS = True
for rotulo, extra in [('base_rag', ['--com-exemplos'])] + ([('base', [])] if AVALIAR_SEM_EXEMPLOS else []):
    subprocess.run([sys.executable, '-m', 'src.finetuning.avaliar', '--modelo', f'hf:{MODELO_BASE_FT}',
                    '--rotulo', rotulo, '--dataset', str(DATASET_FT), '--saida', str(AVALIACAO_FT), *extra],
                   cwd=str(PROJETO), check=True)

## 19. Treinar (QLoRA)

`Qwen3-4B` em 4 bits + LoRA r=16, 1 época (~3 mil pares), perda só no bloco GOBR. Numa T4, conte uns 30 a 60 minutos. Checkpoints vão para o Drive a cada 50 passos: se a sessão cair, rode as etapas 1 a 4, a 17 e esta de novo, e o treino continua do último checkpoint.

In [ ]:
import json

subprocess.run(['pkill', '-f', 'ollama serve'])
subprocess.run([sys.executable, '-m', 'src.finetuning.treinar', '--dataset', str(DATASET_FT),
                '--saida', str(ADAPTADOR_DIR), '--modelo', MODELO_BASE_FT], cwd=str(PROJETO), check=True)

historico = json.loads((ADAPTADOR_DIR / 'historico.json').read_text(encoding='utf-8'))
for registro in historico['historico']:
    if 'loss' in registro or 'eval_loss' in registro:
        print({k: registro[k] for k in ('step', 'loss', 'eval_loss') if k in registro})

## 20. Avaliar o modelo treinado e comparar

Mesmas métricas, mesmos itens. A métrica principal é `ast_igual` (o bloco gerado tem a mesma estrutura da referência); `valido` diz se passa no validador; `vocabulario_ok`, se só usa condições conhecidas.

In [ ]:
for rotulo, extra in [('treinado_rag', ['--com-exemplos']), ('treinado', [])]:
    subprocess.run([sys.executable, '-m', 'src.finetuning.avaliar', '--modelo', f'hf:{MODELO_BASE_FT}',
                    '--adaptador', str(ADAPTADOR_DIR / 'adaptador'), '--rotulo', rotulo,
                    '--dataset', str(DATASET_FT), '--saida', str(AVALIACAO_FT), *extra],
                   cwd=str(PROJETO), check=True)

from IPython.display import Markdown, display
from src.finetuning.avaliar import tabela_comparativa
display(Markdown(tabela_comparativa(AVALIACAO_FT)))

## 20b. Teste final: o agente inteiro, Gerador do Ollama × Gerador treinado

Roda o grafo completo, sem pausar para perguntas, nos 6 casos de `data/avaliacao/casos_agente.json`: os 3 exemplos reais (Maxlub e Infinitto pelos PDFs; V1Vidros, cujo PDF é imagem, pela descrição da tabela), 2 casos escritos com o GOBR esperado e 1 caso com falhas de modelagem de propósito. Compara com o GOBR esperado:

- `valido`: o GOBR final passa no validador e na conferência;
- `cobertura_ast`: blocos esperados com um gerado de mesma estrutura;
- `cobertura_acao`: mesma ação (frequência ou "não agendar") e mesmo limite de dias, aceitando formas equivalentes;
- `achados_ok`: o Analista apontou as falhas que devia (e não apontou as que não existem).

Primeiro com o Gerador servido pelo Ollama (`gemma3:12b`), depois com o `Qwen3-4B` treinado (adaptador da etapa 19). Os outros agentes são iguais nas duas execuções. Conte uns 10 a 20 minutos no total.

In [ ]:
AVALIACAO_AGENTE = Path('/content/drive/MyDrive/gobr-agent/avaliacao_agente')
iniciar_ollama()
base_env = {k: v for k, v in os.environ.items() if k not in ('GOBR_MODELO_GERADOR', 'GOBR_GERADOR_ADAPTADOR')}

execucoes = [('ollama_' + NOME_MODELO, base_env)]
if (ADAPTADOR_DIR / 'adaptador').exists():
    execucoes.append(('gerador_treinado', {**base_env, 'GOBR_MODELO_GERADOR': f'hf:{MODELO_BASE_FT}',
                                           'GOBR_GERADOR_ADAPTADOR': str(ADAPTADOR_DIR / 'adaptador')}))
else:
    print('Adaptador não encontrado: rodando só com o Gerador do Ollama.')

for rotulo, env in execucoes:
    subprocess.run([sys.executable, '-m', 'src.pipelines.avaliar_agente', '--rotulo', rotulo,
                    '--saida', str(AVALIACAO_AGENTE)], cwd=str(PROJETO), env=env, check=True)

from IPython.display import Markdown, display
from src.pipelines.avaliar_agente import tabela
display(Markdown(tabela(AVALIACAO_AGENTE)))

## 21. Usar o Gerador treinado no agente

Aponta o Gerador do grafo para o modelo treinado (Hugging Face, na mesma GPU) e religa o Ollama para os outros agentes. Depois rode as etapas 14 ou 16: o restante do agente não muda. Para voltar ao modelo do Ollama, apague as duas variáveis (ou reinicie a sessão).

In [ ]:
os.environ['GOBR_MODELO_GERADOR'] = f'hf:{MODELO_BASE_FT}'
os.environ['GOBR_GERADOR_ADAPTADOR'] = str(ADAPTADOR_DIR / 'adaptador')
iniciar_ollama()
print('Gerador:', os.environ['GOBR_MODELO_GERADOR'], '+', os.environ['GOBR_GERADOR_ADAPTADOR'])

### Ao retomar outro dia

1. Abra **este mesmo notebook**, ative a GPU, monte o Drive, clone o GitHub e inicie o Ollama (etapas 1 a 6).
2. Rode a etapa **8**: se o explicador não mudou, `DESTINO_LOTE` aponta para a mesma pasta do Drive.
3. Rode a etapa **12** com `EXECUTAR_MAIS = True` até terminar e confira com a **12b**.
4. Baixe `explicacoes.jsonl` (12b). A etapa 13 só confere a recuperação; os índices são refeitos localmente.

Os JSONs salvos no Drive não somem com a desconexão. Itens `failed` são tentados de novo até o limite; `blocked` são regras com trechos que o glossário não traduz (as duas anomalias do acervo).